<center>
  <h1 style="background-color: rgba(128, 253, 255, 0.49); color: rgb(250, 250, 250); padding: 5px; font-size: 30px;">
    <strong> Data Splits Notebook </strong>
  </h1>
</center>

**Student ID's:**

Andreea Roica: 20250361

Jenny Cubelo: 20250431

Libero Biagi: 20250349

Marisa Esteves: 20250348

Oliver Kain: 20250401


#
<h1 style="background-color: rgba(128, 253, 255, 0.49); color: rgb(250, 250, 250); padding: 5px; ; font-size: 30px;">
<strong> Index </strong>
</h1>


[1. **Imports**](#1st-bullet)<br>

[2. **Data Splits**](#2nd-bullet)<br>



#
<h1 id="1st-bullet" style="background-color: rgba(128, 253, 255, 0.49); color: rgb(250, 250, 250); padding: 5px; ; font-size: 30px;">
<strong> 1. Imports </strong>
</h1>

In [ ]:
import tensorflow as tf
print(tf.__version__)
import os
import pandas as pd
from sklearn.model_selection import train_test_split

2.20.0


#
<h1 id="2nd-bullet" style="background-color: rgba(128, 253, 255, 0.49); color: rgb(250, 250, 250); padding: 5px; ; font-size: 30px;">
<strong> 2. Data Splits </strong>
</h1>

We're going to create a csv file that contains metadata about all of our pictures, with the structure _image_path, author_ (which will be our target):

In [19]:
DATASET_DIR = "wikiart" 
OUTPUT_CSV = "dataset/metadata.csv"

data = []

for author in sorted(os.listdir(DATASET_DIR)):
    author_path = os.path.join(DATASET_DIR, author)

    # Skip anything that is not a folder (gives me an error without this)
    if not os.path.isdir(author_path):
        continue
    
    for img_name in sorted(os.listdir(author_path)):
        if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(author_path, img_name)
            data.append({"image_path": img_path, "author": author})

df = pd.DataFrame(data)

df = df.sample(frac=1, random_state=42).reset_index(drop=True) # shuffles the data

df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(df)} entries to {OUTPUT_CSV}")

Saved 13340 entries to dataset/metadata.csv


In [20]:
import os
os.environ["OMP_NUM_THREADS"] = "3"
os.environ["MKL_NUM_THREADS"] = "3"

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import tensorflow as tf

SEED = 42

df = pd.read_csv("dataset/metadata.csv")


def extract_features(path, gray_threshold=0.05):
    img = Image.open(path).convert("RGB")
    img = img.resize((64, 64))
    img = np.asarray(img).astype(np.float32) / 255.0

    rgb_std = np.std(img, axis=2)
    is_gray = float(np.mean(rgb_std) < gray_threshold)

    hsv = tf.image.rgb_to_hsv(img).numpy()

    mean_hsv = hsv.mean(axis=(0, 1))
    std_hsv = hsv.std(axis=(0, 1))

    return pd.Series({
        "is_gray": is_gray,
        "mean_h": mean_hsv[0],
        "mean_s": mean_hsv[1],
        "mean_v": mean_hsv[2],
        "std_h": std_hsv[0],
        "std_s": std_hsv[1],
        "std_v": std_hsv[2],
    })


def choose_k(X, k_min=2, k_max=15, random_state=42):
    n_samples = len(X)

    if n_samples < 2:
        return 1

    k_min = max(2, k_min)
    k_max = min(k_max, n_samples - 1)

    if k_max < k_min:
        return 1

    best_k = k_min
    best_score = -1

    for k in range(k_min, k_max + 1):
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        labels = kmeans.fit_predict(X)

        # silhouette requires at least 3 non-empty clusters
        if len(np.unique(labels)) < 3:
            continue

        score = silhouette_score(X, labels)
        if score > best_score:
            best_score = score
            best_k = k

    return best_k


# Extract image-level features
features = df["image_path"].apply(extract_features)
df = pd.concat([df, features], axis=1)

feature_cols = [
    "is_gray", "mean_h", "mean_s", "mean_v",
    "std_h", "std_s", "std_v"
]

df["cluster"] = -1

for author, idx in df.groupby("author").groups.items():
    author_idx = list(idx)
    author_features = df.loc[author_idx, feature_cols].copy()

    n_images = len(author_features)

    # Scale per author
    X = StandardScaler().fit_transform(author_features)

    # Search range for this author only
    k_max = min(20, max(3, n_images // 40), n_images - 1)
    k = choose_k(X, k_min=2, k_max=k_max, random_state=SEED)

    kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = kmeans.fit_predict(X)

    df.loc[author_idx, "cluster"] = labels

df["cluster"] = df["cluster"].astype(int)
df["artist_cluster"] = df["author"].astype(str) + "_c" + df["cluster"].astype(str)

print(df[["author", "cluster", "artist_cluster"]].head())
print(df["artist_cluster"].value_counts().head(20))

c:\Users\Oliver\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\Oliver\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\Oliver\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\Oliver\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: 

                  author  cluster            artist_cluster
0          Ivan_Shishkin        2          Ivan_Shishkin_c2
1       Camille_Pissarro        2       Camille_Pissarro_c2
2  Pierre_Auguste_Renoir        2  Pierre_Auguste_Renoir_c2
3          Eugene_Boudin        0          Eugene_Boudin_c0
4    John_Singer_Sargent        2    John_Singer_Sargent_c2
artist_cluster
Nicholas_Roerich_c2         833
Vincent_van_Gogh_c2         476
Pierre_Auguste_Renoir_c1    469
Gustave_Dore_c2             468
Pierre_Auguste_Renoir_c2    437
Vincent_van_Gogh_c1         416
Pyotr_Konchalovsky_c1       393
Albrecht_Durer_c0           367
Claude_Monet_c1             358
Pablo_Picasso_c1            334
John_Singer_Sargent_c2      303
Vincent_van_Gogh_c0         299
Claude_Monet_c2             289
Claude_Monet_c0             287
Nicholas_Roerich_c1         285
Rembrandt_c2                273
Camille_Pissarro_c0         266
Martiros_Saryan_c0          242
Camille_Pissarro_c1         236
Marc_Chagall_c2  

c:\Users\Oliver\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


In [21]:
print(df["artist_cluster"].value_counts())

artist_cluster
Nicholas_Roerich_c2         833
Vincent_van_Gogh_c2         476
Pierre_Auguste_Renoir_c1    469
Gustave_Dore_c2             468
Pierre_Auguste_Renoir_c2    437
                           ... 
John_Singer_Sargent_c1       42
Ivan_Shishkin_c3             35
Ilya_Repin_c5                23
Ilya_Repin_c4                19
Gustave_Dore_c1              12
Name: count, Length: 77, dtype: int64


Now we're going to split the data stratifying by author:

In [22]:
SEED = 42

train_val_df, test_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["artist_cluster"],
    random_state=SEED
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1,
    stratify=train_val_df["artist_cluster"],
    random_state=SEED
)

print(len(train_df), len(val_df), len(test_df))

10805 1201 1334


In [ ]:
cols_to_drop = [
    "is_gray", "mean_h", "mean_s", "mean_v",
    "std_h", "std_s", "std_v", "cluster", "artist_cluster"
]

train_df = train_df.drop(columns=cols_to_drop)
val_df   = val_df.drop(columns=cols_to_drop)
test_df  = test_df.drop(columns=cols_to_drop)

In [28]:
train_df.head()

,image_path,author
12086,wikiart\Edgar_Degas\edgar-degas_rider-in-a-red...,Edgar_Degas
12594,wikiart\Pyotr_Konchalovsky\pyotr-konchalovsky_...,Pyotr_Konchalovsky
9988,wikiart\Claude_Monet\claude-monet_lunch-on-the...,Claude_Monet
3646,wikiart\Boris_Kustodiev\boris-kustodiev_blue-h...,Boris_Kustodiev
10478,wikiart\Pablo_Picasso\pablo-picasso_skull-urch...,Pablo_Picasso


Saving the splits:

In [29]:
train_df.to_csv("splits/train.csv", index=False)
val_df.to_csv("splits/val.csv", index=False)
test_df.to_csv("splits/test.csv", index=False)

Everytime that we need to work we will load the csv files from Github like this **(DON'T RESPLIT)**

In [30]:
train_df = pd.read_csv("splits/train.csv")
val_df = pd.read_csv("splits/val.csv")
test_df = pd.read_csv("splits/test.csv")

Previewing the training set:

In [ ]:
train_df

,image_path,author
0,wikiart\Edgar_Degas\edgar-degas_rider-in-a-red...,Edgar_Degas
1,wikiart\Pyotr_Konchalovsky\pyotr-konchalovsky_...,Pyotr_Konchalovsky
2,wikiart\Claude_Monet\claude-monet_lunch-on-the...,Claude_Monet
3,wikiart\Boris_Kustodiev\boris-kustodiev_blue-h...,Boris_Kustodiev
4,wikiart\Pablo_Picasso\pablo-picasso_skull-urch...,Pablo_Picasso
...,...,...
10800,wikiart\Salvador_Dali\salvador-dali_le-char-d-...,Salvador_Dali
10801,wikiart\Vincent_van_Gogh\vincent-van-gogh_mine...,Vincent_van_Gogh
10802,wikiart\Childe_Hassam\childe-hassam_roses.jpg,Childe_Hassam
10803,wikiart\Vincent_van_Gogh\vincent-van-gogh_the-...,Vincent_van_Gogh
